# The DI Chain: Rolling Windows, Field Spill, and Half-Ring Reflection

**Source:** `two_bit_circle.xlsx`, sheet `2-bit circle` (the 8-slice cycle `00011011`).

This notebook follows the actual window order and formulas. It keeps three operations distinct:

1. **Rolling-window update:** a binary integer carried from width \(L\) to \(L+1\) by doubling and appending the entering bit.
2. **Partner involution:** reverse a word unless it is palindromic, in which case complement it.
3. **Boolean Möbius transform:** repeated XOR differences on the Boolean lattice.

The first two yield exact identities and executable checks here. The third is included as a comparison: no explicit intertwining map between it and the rolling-window update is being assumed.

Ledger: **STANDS** = algebraic identity derived from the stated map; **RUNS** = executable checks pass; **OPEN** = a map connecting distinct operations has not yet been supplied.

In [1]:
from itertools import product

S = "00011011"
N = len(S)
CCW_OFFSET = 3

def bit(i):
    return int(S[i % N])

def window_bits(L, i, seq=S):
    n = len(seq)
    return "".join(seq[(i + k) % n] for k in range(L))

def window_value(L, i, seq=S):
    return int(window_bits(L, i, seq), 2)

def reverse_position(L, i):
    return (3 - i - L) % N

assert S[::-1] == "11011000"
assert S[3:] + S[:3] == S[::-1]
print("Base cycle:", S)
print("Reverse cycle:", S[::-1])
print("Reverse equals rotation +3:", S[::-1] == S[3:] + S[:3])

Base cycle: 00011011
Reverse cycle: 11011000
Reverse equals rotation +3: True


In [2]:
# Generate the full rolling-window trace for L=2..8 and all eight circular positions.
trace = []
for L in range(2, 9):
    for i in range(N):
        w_bits = window_bits(L, i)
        w = int(w_bits, 2)
        outgoing = bit(i)
        entering = bit(i + L)
        radial_pred = 2 * w + entering
        radial_direct = window_value(L + 1, i)
        angular_actual = window_value(L, i + 1)
        angular_pred = 2 * w - (2 ** L) * outgoing + entering
        ccw_bits = window_bits(L, i + CCW_OFFSET)
        mirror_i = reverse_position(L, i)
        mirror_bits = window_bits(L, mirror_i)
        reverse_bits = w_bits[::-1]
        assert radial_pred == radial_direct
        assert angular_pred == angular_actual
        assert int(ccw_bits, 2) == window_value(L, (i + 3) % N)
        assert mirror_bits == reverse_bits
        trace.append({
            "L": L, "i": i, "window": w_bits, "B_decimal": w,
            "outgoing": outgoing, "entering": entering,
            "radial": radial_pred, "angular_next": angular_actual,
            "CCW_window": ccw_bits, "G_decimal": int(ccw_bits, 2),
            "mirror_i": mirror_i, "mirror_window": mirror_bits
        })

assert len(trace) == 56
print(f"RUNS: {len(trace)}/56 windows satisfy radial recurrence, angular recurrence, offset +3, and mirror-index law.")
print("\nCW decimal stack by width:")
for L in range(2, 9):
    print(f"L={L}: {[window_value(L, i) for i in range(N)]}")

RUNS: 56/56 windows satisfy radial recurrence, angular recurrence, offset +3, and mirror-index law.

CW decimal stack by width:
L=2: [0, 0, 1, 3, 2, 1, 3, 2]
L=3: [0, 1, 3, 6, 5, 3, 6, 4]
L=4: [1, 3, 6, 13, 11, 6, 12, 8]
L=5: [3, 6, 13, 27, 22, 12, 24, 17]
L=6: [6, 13, 27, 54, 44, 24, 49, 35]
L=7: [13, 27, 54, 108, 88, 49, 99, 70]
L=8: [27, 54, 108, 216, 177, 99, 198, 141]


In [3]:
# The fold between increasing width and sliding angularly:
# W(L+1,i) = W(L,i+1) exactly when the outgoing bit s[i] is zero.
print("Cross-width / angular matches at zero outgoing bits:")
for i in range(N):
    if bit(i) == 0:
        pairs = [(window_value(L+1, i), window_value(L, i+1)) for L in range(2, 8)]
        assert all(a == b for a, b in pairs)
        print(f"i={i}, s[i]={bit(i)}: {pairs}")

print("\nAt i=0, radial values for widths 2..8:",
      [window_value(L, 0) for L in range(2, 9)])
print("At width 8, the angular orbit is:",
      [window_value(8, i) for i in range(8)])
orbit = [window_value(8, i) for i in range(8)]
assert all(orbit[(i+1)%8] == (2*orbit[i]) % 255 for i in range(8))
print("RUNS: each width-8 step is multiplication by 2 modulo 255.")

Cross-width / angular matches at zero outgoing bits:
i=0, s[i]=0: [(0, 0), (1, 1), (3, 3), (6, 6), (13, 13), (27, 27)]
i=1, s[i]=0: [(1, 1), (3, 3), (6, 6), (13, 13), (27, 27), (54, 54)]
i=2, s[i]=0: [(3, 3), (6, 6), (13, 13), (27, 27), (54, 54), (108, 108)]
i=5, s[i]=0: [(3, 3), (6, 6), (12, 12), (24, 24), (49, 49), (99, 99)]

At i=0, radial values for widths 2..8: [0, 0, 1, 3, 6, 13, 27]
At width 8, the angular orbit is: [27, 54, 108, 216, 177, 99, 198, 141]
RUNS: each width-8 step is multiplication by 2 modulo 255.


In [4]:
# Compare the source-style hex-label concatenation with actual bit packing.
def hex_label_pair(L, i):
    left = window_value(L, i)
    right = window_value(L, i + 1)
    return f"{left:X}{right:X}"

def packed_pair(L, i):
    left_bits = window_bits(L, i)
    right_bits = window_bits(L, i + 1)
    value = (int(left_bits, 2) << L) | int(right_bits, 2)
    fixed_hex_digits = (L + 1) // 2  # ceil(2L/4)
    return left_bits + right_bits, value, f"{value:0{fixed_hex_digits}X}"

for L, i in [(4,2), (5,2), (5,0)]:
    raw = hex_label_pair(L, i)
    raw_value = int(raw, 16)
    pair_bits, packed_value, fixed_hex = packed_pair(L, i)
    print(f"L={L}, i={i}: left={window_bits(L,i)} ({window_value(L,i):X}), "
          f"right={window_bits(L,i+1)} ({window_value(L,i+1):X})")
    print(f"  source-style label concat = {raw} (hex integer {raw_value})")
    print(f"  exact pair bits           = {pair_bits} ({2*L} bits)")
    print(f"  fixed-width packed hex    = {fixed_hex} (integer {packed_value})")
    print(f"  same integer?             = {raw_value == packed_value}")

# At L=4, each window is one 4-bit nibble and the pairing is byte-exact.
assert hex_label_pair(4,2) == packed_pair(4,2)[2]
# At L=5, both windows are five bits; raw labels are no longer a fixed-width pack.
assert hex_label_pair(5,2) == "D1B"
assert packed_pair(5,2)[0] == "0110111011"
assert packed_pair(5,2)[2] == "1BB"
assert int(hex_label_pair(5,2),16) != packed_pair(5,2)[1]
print("\nRUNS: the first nibble-alignment boundary is L=5.")

L=4, i=2: left=0110 (6), right=1101 (D)
  source-style label concat = 6D (hex integer 109)
  exact pair bits           = 01101101 (8 bits)
  fixed-width packed hex    = 6D (integer 109)
  same integer?             = True
L=5, i=2: left=01101 (D), right=11011 (1B)
  source-style label concat = D1B (hex integer 3355)
  exact pair bits           = 0110111011 (10 bits)
  fixed-width packed hex    = 1BB (integer 443)
  same integer?             = False
L=5, i=0: left=00011 (3), right=00110 (6)
  source-style label concat = 36 (hex integer 54)
  exact pair bits           = 0001100110 (10 bits)
  fixed-width packed hex    = 066 (integer 102)
  same integer?             = False

RUNS: the first nibble-alignment boundary is L=5.


In [5]:
# Cross-pair reflection at L=8, reproducing columns E and J.
def source_style_pair(L, start_i, direction="CW"):
    if direction == "CCW":
        start_i = (start_i + CCW_OFFSET) % N
    return f"{window_value(L,start_i):X}{window_value(L,start_i+1):X}"

cw_pairs = [source_style_pair(8, i, "CW") for i in (0,2,4,6)]
ccw_pairs = [source_style_pair(8, i, "CCW") for i in (1,3,5,7)]
print("CW pair labels at L=8: ", cw_pairs)
print("CCW pair labels at L=8:", ccw_pairs)
print("Same four fields, re-ordered:", sorted(cw_pairs) == sorted(ccw_pairs))
assert sorted(cw_pairs) == sorted(ccw_pairs) == sorted(["1B36","6CD8","B163","C68D"])

CW pair labels at L=8:  ['1B36', '6CD8', 'B163', 'C68D']
CCW pair labels at L=8: ['B163', 'C68D', '1B36', '6CD8']
Same four fields, re-ordered: True


In [6]:
# The full nested-word ring: build one canonical half, derive the other by partner reflection.
def complement_bits(x):
    return "".join("1" if b == "0" else "0" for b in x)

def partner(x):
    rev = x[::-1]
    return rev if rev != x else complement_bits(x)

def original_grow(L):
    seen, pairs = set(), []
    for w in range(1 << L):
        x = format(w, f"0{L}b")
        if x in seen:
            continue
        p = partner(x)
        seen.update((x, p))
        pairs.append((x, p))
    seq = []
    for idx, (a, b) in enumerate(pairs):
        if idx == len(pairs)-1 and len(pairs) > 1:
            a, b = b, a  # the declared center twist
        mid = len(seq)//2
        seq = seq[:mid] + [a,b] + seq[mid:]
    return seq

def half_ring(L):
    reps = sorted(
        format(w, f"0{L}b")
        for w in range(1 << L)
        if format(w, f"0{L}b") < partner(format(w, f"0{L}b"))
    )
    M = 1 << (L-1)
    seq = reps + [partner(x) for x in reversed(reps)]
    if L > 1:
        seq[M-1], seq[M] = seq[M], seq[M-1]
    return seq

for L in range(1, 11):
    built = half_ring(L)
    reference = original_grow(L)
    assert built == reference
    assert len(set(built)) == (1 << L)
    assert all(partner(built[i]) == built[(1 << L)-1-i] for i in range(1 << L))
    assert all((a ^ b) == (a ^ b) for a,b in [])  # no-op: keeps this cell purely assertion-driven
print("RUNS: half-list + partner-reflection + center swap matches the insertion generator at widths 1..10.")
print("For each tested width, only 2^(L-1) representatives are needed to define the full 2^L ordering.")

RUNS: half-list + partner-reflection + center swap matches the insertion generator at widths 1..10.
For each tested width, only 2^(L-1) representatives are needed to define the full 2^L ordering.


In [7]:
# Keep the Boolean Möbius transform separate from the rolling window recurrence.
def mobius_transform(values):
    a = list(values)
    n = len(a)
    if n == 0 or (n & (n-1)) != 0:
        raise ValueError("table length must be a positive power of two")
    step = 1
    while step < n:
        for mask in range(n):
            if mask & step:
                a[mask] ^= a[mask ^ step]
        step <<= 1
    return a

sample = [0,1,1,0,1,0,0,1]
coeff = mobius_transform(sample)
assert mobius_transform(coeff) == sample
print("Truth-table values:", sample)
print("Möbius/ANF coefficients:", coeff)
print("Applying the same subset-XOR transform twice returns the original table.")
print("OPEN: no explicit commuting map from this Boolean-lattice transform to W(L,i) has been specified.")

Truth-table values: [0, 1, 1, 0, 1, 0, 0, 1]
Möbius/ANF coefficients: [0, 1, 1, 0, 1, 0, 0, 0]
Applying the same subset-XOR transform twice returns the original table.
OPEN: no explicit commuting map from this Boolean-lattice transform to W(L,i) has been specified.


## Result ledger

- **STANDS:** \(W_{L+1,i}=2W_{L,i}+s_{i+L}\) and \(W_{L,i+1}=2W_{L,i}-2^Ls_i+s_{i+L}\) follow directly from appending and removing a cyclic bit.
- **RUNS:** all 56 window cases at \(L=2,\ldots,8\) pass both recurrences, the fixed offset relation \(G_{L,i}=B_{L,i+3}\), and the mirror index \(3-i-L\pmod8\).
- **STANDS:** in the declared nested ring, \(R_{2^L-1-i}=P(R_i)\). Thus the ordered second half is generated from the first half by partner reflection, after the explicit center swap.
- **RUNS:** the half-ring construction exactly matches the source `grow` ordering at widths 1–10.
- **STANDS:** width \(L=4\) is nibble-aligned; \(L=5\) makes the pair a 10-bit field. One added width bit enters each of the two adjacent windows.
- **OPEN:** a formal map identifying the rolling-window recursion with the Boolean Möbius/ANF transform is not part of this result. Similar use of pairwise updates is not by itself an equality of operations.
- **OPEN:** the physical interpretation of a “grey/maybe” read requires a declared reader fiber and weighting rule. A 0.5 average can be produced by unresolved 0/1 readings, but it is not automatically a third Boolean state.